In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
import warnings
warnings.filterwarnings("ignore")

# Load saved models and scaler
xgb_model = joblib.load("../models/xgb_model.pkl")
lr_model   = joblib.load("../models/lr_model.pkl")
scaler     = joblib.load("../models/scaler.pkl")

# Reload and reprocess dataset (same steps as before)
df = pd.read_csv("../data/german_credit.csv")
df_clean = df.copy()
df_clean["Risk"] = df_clean["Risk"].map({"good": 0, "bad": 1})

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cat_cols = df_clean.select_dtypes(include="object").columns
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

X = df_clean.drop("Risk", axis=1)
y = df_clean["Risk"]

X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns)

print("✅ Models and data loaded successfully!")
print(f"Dataset shape: {X_scaled.shape}")

✅ Models and data loaded successfully!
Dataset shape: (1000, 10)


In [2]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_scaled)

print("✅ SHAP explainer ready!")
print(f"SHAP values shape: {shap_values.shape}")

✅ SHAP explainer ready!
SHAP values shape: (1000, 10)


In [3]:
def plain_language_explanation(shap_vals, feature_names, top_n=3):
    """
    Converts SHAP values into human-readable sentences.
    Returns a list of explanation strings for the top N features.
    """
    # Map feature names to readable labels
    feature_labels = {
        "Checking account"  : "checking account status",
        "Duration"          : "loan duration",
        "Credit amount"     : "credit amount requested",
        "Saving accounts"   : "savings account status",
        "Age"               : "age",
        "Purpose"           : "loan purpose",
        "Housing"           : "housing situation",
        "Sex"               : "gender",
        "Job"               : "job type",
        "Unnamed: 0"        : "applicant record index"
    }

    # Pair each feature with its SHAP value
    shap_pairs = list(zip(feature_names, shap_vals))
    
    # Sort by absolute impact (highest first)
    shap_pairs_sorted = sorted(shap_pairs, key=lambda x: abs(x[1]), reverse=True)
    
    explanations = []
    for feature, shap_val in shap_pairs_sorted[:top_n]:
        label = feature_labels.get(feature, feature)
        direction = "increased" if shap_val > 0 else "reduced"
        strength  = "significantly" if abs(shap_val) > 0.5 else "moderately" if abs(shap_val) > 0.2 else "slightly"
        
        sentence = (f"Your {label} {strength} {direction} your credit risk "
                   f"(impact score: {shap_val:+.3f}).")
        explanations.append({
            "feature"  : feature,
            "label"    : label,
            "shap_val" : shap_val,
            "direction": direction,
            "strength" : strength,
            "sentence" : sentence
        })
    
    return explanations

# Test on applicant 0
applicant_idx  = 0
applicant_shap = shap_values[applicant_idx]
explanations   = plain_language_explanation(applicant_shap, X_scaled.columns.tolist())

print("=== Plain-Language Explanation ===\n")
for i, exp in enumerate(explanations):
    print(f"  {i+1}. {exp['sentence']}")

=== Plain-Language Explanation ===

  1. Your loan duration significantly reduced your credit risk (impact score: -2.070).
  2. Your checking account status significantly increased your credit risk (impact score: +0.517).
  3. Your age moderately reduced your credit risk (impact score: -0.305).


In [4]:
def recommendation_engine(shap_vals, feature_names, feature_values):
    """
    Generates actionable financial improvement tips
    based on which features most negatively impacted the score.
    """
    # Rules mapped to each feature
    rules = {
        "Checking account": {
            "tip"      : "Open or maintain an active checking account with a positive balance.",
            "reason"   : "Having little or no checking account balance is the strongest predictor of credit risk in this model."
        },
        "Duration": {
            "tip"      : "Consider applying for a shorter loan duration if possible.",
            "reason"   : "Longer loan durations increase perceived repayment risk."
        },
        "Credit amount": {
            "tip"      : "Consider applying for a lower credit amount relative to your income.",
            "reason"   : "Higher loan amounts increase the likelihood of default risk."
        },
        "Saving accounts": {
            "tip"      : "Build up your savings account balance before reapplying.",
            "reason"   : "Applicants with little or no savings are considered higher risk."
        },
        "Age": {
            "tip"      : "Younger applicants may benefit from building a credit history first.",
            "reason"   : "Age is associated with financial stability and repayment track record."
        },
        "Purpose": {
            "tip"      : "Loan purpose affects risk assessment — essential purposes (education, car) are viewed more favourably.",
            "reason"   : "Certain loan purposes are statistically associated with higher default rates."
        },
        "Housing": {
            "tip"      : "Owning property is viewed positively in credit assessments.",
            "reason"   : "Housing stability is an indicator of financial responsibility."
        },
        "Sex": {
            "tip"      : "Gender should not affect your application — this is monitored for fairness.",
            "reason"   : "This feature is tracked to ensure the model does not discriminate unfairly."
        },
        "Job": {
            "tip"      : "Stable, skilled employment improves your credit profile.",
            "reason"   : "Employment type and stability are linked to repayment reliability."
        },
    }

    # Get features that pushed toward BAD credit (positive SHAP = bad)
    shap_pairs = list(zip(feature_names, shap_vals))
    negative_factors = [
        (feat, val) for feat, val in shap_pairs 
        if val > 0 and feat in rules  # positive SHAP = pushes toward bad credit
    ]
    
    # Sort by highest negative impact
    negative_factors_sorted = sorted(negative_factors, key=lambda x: x[1], reverse=True)
    
    tips = []
    for rank, (feature, shap_val) in enumerate(negative_factors_sorted[:3], start=1):
        rule = rules[feature]
        tips.append({
            "rank"      : rank,
            "feature"   : feature,
            "shap_val"  : shap_val,
            "tip"       : rule["tip"],
            "reason"    : rule["reason"],
            "priority"  : "High" if shap_val > 0.5 else "Medium" if shap_val > 0.2 else "Low"
        })
    
    return tips

# Test on applicant 0
feature_values = X_scaled.iloc[applicant_idx].to_dict()
tips = recommendation_engine(applicant_shap, X_scaled.columns.tolist(), feature_values)

print("=== Financial Improvement Recommendations ===\n")
for tip in tips:
    print(f"  [{tip['priority']} PRIORITY] #{tip['rank']} — {tip['feature']}")
    print(f"  💡 Tip    : {tip['tip']}")
    print(f"  📌 Reason : {tip['reason']}")
    print(f"  📊 Impact : {tip['shap_val']:+.3f}\n")

=== Financial Improvement Recommendations ===

  [High PRIORITY] #1 — Checking account
  💡 Tip    : Open or maintain an active checking account with a positive balance.
  📌 Reason : Having little or no checking account balance is the strongest predictor of credit risk in this model.
  📊 Impact : +0.517



In [5]:
def simulate_approval(applicant_idx, feature_to_change, new_value, 
                      X_scaled, scaler, xgb_model):
    """
    Simulates how changing a single financial parameter
    affects the applicant's approval probability.
    """
    # Copy the applicant's current data
    applicant_data = X_scaled.iloc[[applicant_idx]].copy()
    
    # Current probability
    current_prob_bad  = xgb_model.predict_proba(applicant_data)[0][1]
    current_prob_good = 1 - current_prob_bad
    
    # Apply the change
    applicant_data[feature_to_change] = new_value
    
    # New probability after change
    new_prob_bad  = xgb_model.predict_proba(applicant_data)[0][1]
    new_prob_good = 1 - new_prob_bad
    
    # Calculate improvement
    improvement = (new_prob_good - current_prob_good) * 100
    
    return {
        "feature_changed"       : feature_to_change,
        "current_approval_prob" : round(current_prob_good * 100, 2),
        "new_approval_prob"     : round(new_prob_good * 100, 2),
        "improvement"           : round(improvement, 2),
        "current_decision"      : "✅ Approved" if current_prob_good >= 0.5 else "❌ Rejected",
        "new_decision"          : "✅ Approved" if new_prob_good >= 0.5 else "❌ Rejected",
    }

# ── Simulation examples ──────────────────────────────────────────────────
applicant_idx = 0

print("=" * 55)
print(f"  LIVE SIMULATOR — Applicant #{applicant_idx}")
print("=" * 55)

# Simulate 1: Improve checking account (set to high value)
sim1 = simulate_approval(applicant_idx, "Checking account", 
                         2.0, X_scaled, scaler, xgb_model)
print(f"\n📊 Scenario 1: Improve Checking Account")
print(f"   Before : {sim1['current_approval_prob']}% ({sim1['current_decision']})")
print(f"   After  : {sim1['new_approval_prob']}% ({sim1['new_decision']})")
print(f"   Change : {sim1['improvement']:+.2f}%")

# Simulate 2: Reduce loan duration (set to lower scaled value)
sim2 = simulate_approval(applicant_idx, "Duration", 
                         -1.0, X_scaled, scaler, xgb_model)
print(f"\n📊 Scenario 2: Reduce Loan Duration")
print(f"   Before : {sim2['current_approval_prob']}% ({sim2['current_decision']})")
print(f"   After  : {sim2['new_approval_prob']}% ({sim2['new_decision']})")
print(f"   Change : {sim2['improvement']:+.2f}%")

# Simulate 3: Increase savings (set to high value)
sim3 = simulate_approval(applicant_idx, "Saving accounts", 
                         2.0, X_scaled, scaler, xgb_model)
print(f"\n📊 Scenario 3: Increase Savings")
print(f"   Before : {sim3['current_approval_prob']}% ({sim3['current_decision']})")
print(f"   After  : {sim3['new_approval_prob']}% ({sim3['new_decision']})")
print(f"   Change : {sim3['improvement']:+.2f}%")

  LIVE SIMULATOR — Applicant #0

📊 Scenario 1: Improve Checking Account
   Before : 95.45999908447266% (✅ Approved)
   After  : 98.83999633789062% (✅ Approved)
   Change : +3.38%

📊 Scenario 2: Reduce Loan Duration
   Before : 95.45999908447266% (✅ Approved)
   After  : 89.62999725341797% (✅ Approved)
   Change : -5.83%

📊 Scenario 3: Increase Savings
   Before : 95.45999908447266% (✅ Approved)
   After  : 95.45999908447266% (✅ Approved)
   Change : +0.00%


In [8]:
def full_applicant_report(applicant_idx, X_scaled, shap_values, xgb_model):
    """
    Generates a complete credit assessment report for one applicant
    combining score, plain-language explanation, and recommendations.
    """
    applicant_data = X_scaled.iloc[[applicant_idx]]
    applicant_shap = shap_values[applicant_idx]
    feature_values = X_scaled.iloc[applicant_idx].to_dict()
    
    # Score
    prob_bad   = xgb_model.predict_proba(applicant_data)[0][1]
    prob_good  = 1 - prob_bad
    credit_score = round(prob_good * 100, 1)
    decision   = "APPROVED ✅" if prob_good >= 0.5 else "REJECTED ❌"
    risk_class = "Low Risk" if prob_good >= 0.7 else "Medium Risk" if prob_good >= 0.5 else "High Risk"
    
    # Explanations
    explanations = plain_language_explanation(
        applicant_shap, X_scaled.columns.tolist(), top_n=3)
    
    # Tips
    tips = recommendation_engine(
        applicant_shap, X_scaled.columns.tolist(), feature_values)
    
    print("=" * 60)
    print(f"  CREDIT ASSESSMENT REPORT — Applicant #{applicant_idx}")
    print("=" * 60)
    print(f"\n  Credit Score    : {credit_score}/100")
    print(f"  Decision        : {decision}")
    print(f"  Risk Class      : {risk_class}")
    print(f"  Approval Prob   : {prob_good*100:.1f}%")
    
    print(f"\n  {'─'*55}")
    print(f"  WHY THIS DECISION? (Top 3 Factors)")
    print(f"  {'─'*55}")
    for exp in explanations:
        print(f"  • {exp['sentence']}")
    
    print(f"\n  {'─'*55}")
    print(f"  HOW TO IMPROVE YOUR SCORE")
    print(f"  {'─'*55}")
    if tips:
        for tip in tips:
            print(f"  [{tip['priority']}] {tip['tip']}")
    else:
        print("  Your profile looks strong — keep maintaining good financial habits!")
    
    print(f"\n{'='*60}\n")

# Generate report for first 3 applicants
for idx in [0, 1, 2]:
    full_applicant_report(idx, X_scaled, shap_values, xgb_model)

  CREDIT ASSESSMENT REPORT — Applicant #0

  Credit Score    : 95.5/100
  Decision        : APPROVED ✅
  Risk Class      : Low Risk
  Approval Prob   : 95.5%

  ───────────────────────────────────────────────────────
  WHY THIS DECISION? (Top 3 Factors)
  ───────────────────────────────────────────────────────
  • Your loan duration significantly reduced your credit risk (impact score: -2.070).
  • Your checking account status significantly increased your credit risk (impact score: +0.517).
  • Your age moderately reduced your credit risk (impact score: -0.305).

  ───────────────────────────────────────────────────────
  HOW TO IMPROVE YOUR SCORE
  ───────────────────────────────────────────────────────
  [High] Open or maintain an active checking account with a positive balance.


  CREDIT ASSESSMENT REPORT — Applicant #1

  Credit Score    : 47.29999923706055/100
  Decision        : REJECTED ❌
  Risk Class      : High Risk
  Approval Prob   : 47.3%

  ───────────────────────────────

In [10]:
import os

os.makedirs("../src", exist_ok=True)

xai_code = '''
import numpy as np
import joblib
import shap

def load_models():
    xgb_model = joblib.load("models/xgb_model.pkl")
    scaler     = joblib.load("models/scaler.pkl")
    return xgb_model, scaler

def get_shap_values(xgb_model, X_scaled):
    explainer  = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_scaled)
    return explainer, shap_values

def plain_language_explanation(shap_vals, feature_names, top_n=3):
    feature_labels = {
        "Checking account"  : "checking account status",
        "Duration"          : "loan duration",
        "Credit amount"     : "credit amount requested",
        "Saving accounts"   : "savings account status",
        "Age"               : "age",
        "Purpose"           : "loan purpose",
        "Housing"           : "housing situation",
        "Sex"               : "gender",
        "Job"               : "job type",
    }
    shap_pairs  = list(zip(feature_names, shap_vals))
    shap_sorted = sorted(shap_pairs, key=lambda x: abs(x[1]), reverse=True)
    explanations = []
    for feature, shap_val in shap_sorted[:top_n]:
        label     = feature_labels.get(feature, feature)
        direction = "increased" if shap_val > 0 else "reduced"
        strength  = "significantly" if abs(shap_val) > 0.5 else "moderately" if abs(shap_val) > 0.2 else "slightly"
        explanations.append({
            "feature"  : feature,
            "label"    : label,
            "shap_val" : shap_val,
            "direction": direction,
            "sentence" : f"Your {label} {strength} {direction} your credit risk (impact: {shap_val:+.3f})."
        })
    return explanations

def recommendation_engine(shap_vals, feature_names):
    rules = {
        "Checking account": {"tip": "Open or maintain an active checking account with a positive balance.", "reason": "Checking account status is the strongest predictor of credit risk."},
        "Duration"        : {"tip": "Consider applying for a shorter loan duration.", "reason": "Longer durations increase repayment risk."},
        "Credit amount"   : {"tip": "Apply for a lower credit amount relative to your income.", "reason": "Higher amounts increase default risk."},
        "Saving accounts" : {"tip": "Build up your savings before reapplying.", "reason": "Low savings indicates financial vulnerability."},
        "Age"             : {"tip": "Build a credit history through smaller loans first.", "reason": "Age reflects financial experience and stability."},
        "Purpose"         : {"tip": "Essential loan purposes are viewed more favourably.", "reason": "Loan purpose is associated with repayment intent."},
        "Housing"         : {"tip": "Property ownership improves your credit profile.", "reason": "Housing stability indicates financial responsibility."},
        "Job"             : {"tip": "Stable skilled employment improves your credit profile.", "reason": "Employment type reflects repayment reliability."},
    }
    shap_pairs   = list(zip(feature_names, shap_vals))
    neg_factors  = sorted([(f,v) for f,v in shap_pairs if v > 0 and f in rules], key=lambda x: x[1], reverse=True)
    tips = []
    for rank, (feature, shap_val) in enumerate(neg_factors[:3], start=1):
        rule = rules[feature]
        tips.append({
            "rank"    : rank,
            "feature" : feature,
            "shap_val": shap_val,
            "tip"     : rule["tip"],
            "reason"  : rule["reason"],
            "priority": "High" if shap_val > 0.5 else "Medium" if shap_val > 0.2 else "Low"
        })
    return tips

def simulate_approval(applicant_data, feature_to_change, new_value, xgb_model):
    current_prob_good = 1 - xgb_model.predict_proba(applicant_data)[0][1]
    modified_data     = applicant_data.copy()
    modified_data[feature_to_change] = new_value
    new_prob_good     = 1 - xgb_model.predict_proba(modified_data)[0][1]
    return {
        "feature_changed"       : feature_to_change,
        "current_approval_prob" : round(current_prob_good * 100, 2),
        "new_approval_prob"     : round(new_prob_good * 100, 2),
        "improvement"           : round((new_prob_good - current_prob_good) * 100, 2),
        "current_decision"      : "Approved" if current_prob_good >= 0.5 else "Rejected",
        "new_decision"          : "Approved" if new_prob_good >= 0.5 else "Rejected",
    }
'''

with open("../src/xai_engine.py", "w") as f:
    f.write(xai_code)

print("✅ XAI engine saved to src/xai_engine.py")
print("✅ Ready for Flask API integration in Phase 3!")

✅ XAI engine saved to src/xai_engine.py
✅ Ready for Flask API integration in Phase 3!


In [1]:
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder

scaler = joblib.load("../models/scaler.pkl")
df = pd.read_csv("../data/german_credit.csv")

le = LabelEncoder()
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("=== Scaler means (centre values) ===")
for col, mean, std in zip(df.columns, scaler.mean_, scaler.scale_):
    print(f"  {col:25s}: mean={mean:.3f}  std={std:.3f}")

print("\n=== Checking account value counts ===")
print(df["Checking account"].value_counts().sort_index())

print("\n=== Saving accounts value counts ===")
print(df["Saving accounts"].value_counts().sort_index())

print("\n=== Risk value counts ===")
df2 = pd.read_csv("../data/german_credit.csv")
df2["Risk"] = df2["Risk"].map({"good": 0, "bad": 1})
print(df2["Risk"].value_counts())

print("\n=== Checking account vs Risk ===")
df["Risk"] = df2["Risk"]
print(df.groupby("Checking account")["Risk"].mean().round(3))

print("\n=== Saving accounts vs Risk ===")
print(df.groupby("Saving accounts")["Risk"].mean().round(3))

=== Scaler means (centre values) ===
  Unnamed: 0               : mean=499.500  std=288.675
  Age                      : mean=35.546  std=11.370
  Sex                      : mean=0.690  std=0.462
  Job                      : mean=1.904  std=0.653
  Housing                  : mean=1.071  std=0.531
  Saving accounts          : mean=1.105  std=1.579
  Checking account         : mean=1.577  std=1.257
  Credit amount            : mean=3271.258  std=2821.325
  Duration                 : mean=20.903  std=12.053
  Purpose                  : mean=2.878  std=1.977

=== Checking account value counts ===
Checking account
0    274
1    269
2     63
3    394
Name: count, dtype: int64

=== Saving accounts value counts ===
Saving accounts
0    603
1    103
2     63
3     48
4    183
Name: count, dtype: int64

=== Risk value counts ===
Risk
0    700
1    300
Name: count, dtype: int64

=== Checking account vs Risk ===
Checking account
0    0.493
1    0.390
2    0.222
3    0.117
Name: Risk, dtype: float6

C:\Users\User\AppData\Local\Temp\ipykernel_89276\951583597.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns
